# IJMLC English Extension — Ontology-v2 Pilot Runner v7

This notebook runs the **second English pilot** designed to isolate the contribution of
the ontology itself.

It compares, on the same RoBERTa backbone and seed:

- **Baseline** — text only.
- **NRC-flat** — 7-d lexical emotion vector, no ontology structure.
- **RawGate-Zero** — RawGate architecture but the 24-d ontology vector is all zeros.
- **RawGate-Shuffled** — RawGate with real ontology vectors randomly reassigned between
  samples within each split.
- **RawGate-v2** — the proposed RawGate with correctly aligned ontology-v2 vectors.

The English lexical layer is built from NRC emotion/polarity evidence plus
**appraisals inferred only from explicit relations in
`ekman_appraisal_ontology_v2.rdf`**. Intensity is not inferred because NRC does not
supply intensity evidence.

The run also produces:

- gate diagnostics,
- 24-d feature-dimension audit,
- ontology/appraisal coverage metrics,
- ontology-active/inactive evaluation,
- appraisal-active/inactive evaluation.

## Kaggle settings

- **GPU:** ON
- **Internet:** ON
- Attach two inputs:
  1. `ijmlc_english_extension_package_v2` (the package supplied with this notebook)
  2. your private NRC Emotion Lexicon dataset

The notebook does **not** run five seeds by default. First inspect the second pilot.

In [ ]:
# ============================================================
# 1. USER CONTROLS
# ============================================================

RUN_PILOT_V2 = True

# Keep False until the second pilot has been reviewed.
RUN_CONFIRMATORY_5SEED = False

PILOT_SEED = [42]
FULL_SEEDS = [0, 42, 123, 1234, 2025]

PILOT_MODELS = [
    "baseline",
    "nrcflat",
    "rawgate_zero",
    "rawgate_shuffled",
    "rawgate",
]

# A smaller confirmatory set, intentionally excluding secondary fusion variants.
CONFIRMATORY_MODELS = [
    "baseline",
    "rawgate_zero",
    "rawgate_shuffled",
    "rawgate",
]

# Keep None to preserve the package YAML value (32).
# If Kaggle reports CUDA OOM, change ONLY this value to 16.
BATCH_SIZE_OVERRIDE = None

REPO_URL = (
    "https://github.com/minkduck/"
    "KG-BERT-Vietnamese-Emotion-Classification.git"
)
REPO_BRANCH = "main"

print("RUN_PILOT_V2           =", RUN_PILOT_V2)
print("RUN_CONFIRMATORY_5SEED =", RUN_CONFIRMATORY_5SEED)
print("PILOT_MODELS           =", PILOT_MODELS)
print("BATCH_SIZE_OVERRIDE    =", BATCH_SIZE_OVERRIDE)

In [ ]:
# ============================================================
# 2. ENVIRONMENT + AUTOMATIC INPUT DISCOVERY
# ============================================================

import os
import sys
import json
import shutil
import zipfile
import subprocess
import urllib.request
from pathlib import Path

# Recover safely even if a previous Kaggle run deleted its old working directory.
os.chdir("/kaggle/working")

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

REPO_DIR = WORK_ROOT / "KG-BERT-Vietnamese-Emotion-Classification"
DATA_ROOT = WORK_ROOT / "data" / "english"
RESULT_ROOT = WORK_ROOT / "results" / "english"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Input root:", INPUT_ROOT)
print("Working root:", WORK_ROOT)

# ---- NRC auto-discovery -------------------------------------------------
nrc_candidates = sorted(
    INPUT_ROOT.rglob("NRC-Emotion-Lexicon-Wordlevel-v0.92.txt")
)
if not nrc_candidates:
    nrc_candidates = sorted(
        p for p in INPUT_ROOT.rglob("*")
        if p.is_file()
        and "NRC-Emotion-Lexicon" in p.name
        and p.suffix.lower() in {".txt", ".tsv", ".csv"}
    )
if not nrc_candidates:
    raise FileNotFoundError(
        "NRC Emotion Lexicon was not found below /kaggle/input. "
        "Attach the private NRC dataset first."
    )
NRC_PATH = nrc_candidates[0]

# ---- Package-v2 auto-discovery -----------------------------------------
manifest_candidates = sorted(
    INPUT_ROOT.rglob("PACKAGE_V2_MANIFEST.json")
)
package_roots = []

for manifest in manifest_candidates:
    root = manifest.parent
    required = [
        root / "ontology" / "ekman_appraisal_ontology_v2.rdf",
        root / "scripts" / "build_english_lexicon.py",
        root / "scripts" / "run_experiment.py",
        root / "src" / "language_adapters.py",
        root / "src" / "ontology_engine.py",
        root / "src" / "models" / "transformer_fusion_models.py",
        root / "configs" / "dailydialog_roberta.yaml",
        root / "configs" / "meld_roberta.yaml",
    ]
    if all(p.exists() for p in required):
        package_roots.append(root)

PACKAGE_ROOT = package_roots[0] if package_roots else None
PACKAGE_ZIP = None

# Fallback if Kaggle keeps the uploaded package as ZIP rather than extracting it.
if PACKAGE_ROOT is None:
    zip_candidates = sorted(
        p for p in INPUT_ROOT.rglob("*.zip")
        if "ijmlc" in p.name.lower()
        and "extension" in p.name.lower()
        and ("v2" in p.name.lower() or "package" in p.name.lower())
    )
    if zip_candidates:
        PACKAGE_ZIP = zip_candidates[0]

if PACKAGE_ROOT is None and PACKAGE_ZIP is None:
    raise FileNotFoundError(
        "Cannot find the IJMLC package v2. Attach the supplied "
        "ijmlc_english_extension_package_v2.zip as a Kaggle input."
    )

print("\n=== INPUT DISCOVERY ===")
print("NRC_PATH     :", NRC_PATH)
print("NRC exists   :", NRC_PATH.exists())
print("PACKAGE_ROOT :", PACKAGE_ROOT)
print("PACKAGE_ZIP  :", PACKAGE_ZIP)

assert NRC_PATH.exists()

# ---- GPU check ----------------------------------------------------------
try:
    import torch
    print("\n=== GPU CHECK ===")
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: CUDA is OFF. Enable GPU before training.")
except Exception as exc:
    print("Torch check warning:", exc)

In [ ]:
# ============================================================
# 3. INTERNET CHECK, CLONE REPOSITORY, OVERLAY THE PACKAGE
#    v5: robust against a deleted/stale current working directory
# ============================================================

# IMPORTANT:
# When a previous Kaggle run ended with the process working inside REPO_DIR
# and the next run deletes that directory, the Python process can be left
# with a non-existent current working directory. Then `git clone` fails with:
#   fatal: Unable to read current working directory: No such file or directory
#
# Fix: always move to the stable Kaggle working root BEFORE deleting/recloning.

# Force a valid, stable current working directory first.
os.chdir("/kaggle/working")
WORK_ROOT = Path("/kaggle/working").resolve()
REPO_DIR = WORK_ROOT / "KG-BERT-Vietnamese-Emotion-Classification"

print("Stable working directory:", os.getcwd())

def check_url(url, timeout=20):
    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            return True, getattr(response, "status", 200)
    except Exception as e:
        return False, str(e)

print("\n=== INTERNET CHECK ===")
for url in [
    "https://github.com",
    "https://aclanthology.org",
    "https://huggingface.co",
]:
    ok, status = check_url(url)
    print(f"{url:35s} -> {status}")
    if not ok:
        raise RuntimeError(
            "Internet access is required. Turn Internet ON in Kaggle Settings "
            "and rerun this cell."
        )

# Remove an old working copy only AFTER moving out of it.
if REPO_DIR.exists():
    print("\nRemoving previous working repository:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

# Re-confirm cwd is valid after cleanup.
os.chdir(WORK_ROOT)
print("Current directory before clone:", os.getcwd())

print("\nCloning repository...")
clone = subprocess.run(
    [
        "git", "clone",
        "--depth", "1",
        "--branch", REPO_BRANCH,
        REPO_URL,
        str(REPO_DIR),
    ],
    cwd=str(WORK_ROOT),
    text=True,
    capture_output=True,
)

# Show git output clearly.
if clone.stdout:
    print(clone.stdout)
if clone.stderr:
    print(clone.stderr)

if clone.returncode != 0:
    raise RuntimeError(
        "git clone failed with return code "
        f"{clone.returncode}. See output above."
    )

if not (REPO_DIR / ".git").exists():
    raise RuntimeError(
        "Repository clone did not create a valid .git directory."
    )

# Resolve package root if the input was a ZIP rather than an extracted dataset.
if PACKAGE_ROOT is None:
    overlay_dir = WORK_ROOT / "ijmlc_overlay"

    # Again ensure we are not inside the directory being deleted.
    os.chdir(WORK_ROOT)

    if overlay_dir.exists():
        shutil.rmtree(overlay_dir)

    overlay_dir.mkdir(parents=True, exist_ok=True)

    print("Extracting package ZIP:", PACKAGE_ZIP)
    with zipfile.ZipFile(PACKAGE_ZIP, "r") as z:
        z.extractall(overlay_dir)

    candidate_readmes = sorted(overlay_dir.rglob("PACKAGE_V2_MANIFEST.json"))
    if not candidate_readmes:
        raise RuntimeError(
            "PACKAGE_V2_MANIFEST.json was not found after extracting the package ZIP."
        )

    PACKAGE_ROOT = candidate_readmes[0].parent

print("Overlay package root:", PACKAGE_ROOT)

# Copy extension files over the current GitHub main branch.
shutil.copytree(
    PACKAGE_ROOT,
    REPO_DIR,
    dirs_exist_ok=True,
)

# Enter repository only after clone + overlay are complete.
os.chdir(REPO_DIR)

print("\nRepository ready at:", os.getcwd())
print("\nGit status after overlay:")
subprocess.run(
    ["git", "status", "--short"],
    cwd=str(REPO_DIR),
    check=False,
)

print("\nCLONE + OVERLAY PASSED.")

In [ ]:
# ============================================================
# 4. VERIFY PACKAGE V2 FILES
# ============================================================

required_files = [
    "PACKAGE_V2_MANIFEST.json",
    "README_KAGGLE_V2.md",
    "src/language_adapters.py",
    "src/data_loading.py",
    "src/ontology_engine.py",
    "src/models/transformer_fusion_models.py",
    "src/models/__init__.py",
    "scripts/build_english_lexicon.py",
    "scripts/run_experiment.py",
    "configs/dailydialog_roberta.yaml",
    "configs/meld_roberta.yaml",
    "ontology/ekman_appraisal_ontology_v2.rdf",
    "requirements_english.txt",
    "requirements.txt",
]

all_ok = True
for rel in required_files:
    p = REPO_DIR / rel
    ok = p.exists()
    print(f"{'OK' if ok else 'MISSING':8s} {rel}")
    all_ok = all_ok and ok

if not all_ok:
    raise RuntimeError(
        "Some package-v2 files are missing. Do not continue to training."
    )

print("\nALL PACKAGE-V2 FILES ARE PRESENT.")

In [ ]:
# ============================================================
# 5. INSTALL DEPENDENCIES — KAGGLE-SAFE VERSION FOR PILOT V2
# ============================================================
#
# IMPORTANT:
# The repository's historical requirements.txt pins:
#     protobuf==3.20.3
# That version is too old for several packages preinstalled in the current
# Kaggle image (e.g. grpcio-tools / Google Cloud packages).
#
# For Kaggle we therefore:
#   1) preserve the notebook runtime packages,
#   2) ignore the legacy protobuf/ipykernel/notebook pins,
#   3) install protobuf 5.x, which satisfies the current Kaggle stack,
#   4) install only the project dependencies that are actually needed.
#
# This changes the execution environment only; it does NOT change the
# scientific model, datasets, seeds, hyperparameters, or evaluation protocol.

os.chdir(REPO_DIR)

original_req = REPO_DIR / "requirements.txt"
safe_req = WORK_ROOT / "requirements_kaggle_safe.txt"

skip_prefixes = (
    "protobuf",
    "ipykernel",
    "notebook",
)

kept_lines = []
skipped_lines = []

for raw_line in original_req.read_text(encoding="utf-8").splitlines():
    line = raw_line.strip()

    # Keep comments and blank lines.
    if not line or line.startswith("#"):
        kept_lines.append(raw_line)
        continue

    normalized = line.lower().replace(" ", "")
    package_name = (
        normalized
        .split("==")[0]
        .split(">=")[0]
        .split("<=")[0]
        .split("~=")[0]
        .split(">")[0]
        .split("<")[0]
    )

    if package_name in skip_prefixes:
        skipped_lines.append(raw_line)
    else:
        kept_lines.append(raw_line)

safe_req.write_text(
    "\n".join(kept_lines) + "\n",
    encoding="utf-8",
)

print("Kaggle-safe requirements created:")
print(safe_req)

print("\nSkipped legacy/runtime-sensitive requirements:")
for line in skipped_lines:
    print("  -", line)

# First restore a protobuf version compatible with the current Kaggle image.
# protobuf 5.29.x satisfies the conflicts shown by the current Kaggle runtime:
# grpcio-tools requires >=5.26.1,<6 and a2a-sdk requires >=5.29.5.
print("\nInstalling Kaggle-compatible protobuf...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "protobuf>=5.29.5,<6",
    ],
    check=True,
)

print("Installing repository dependencies (without legacy protobuf/kernel pins)...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(safe_req),
    ],
    check=True,
)

print("Installing English-extension dependencies...")
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements_english.txt",
    ],
    check=True,
)

# Optional batch-size override.
if BATCH_SIZE_OVERRIDE is not None:
    import yaml

    for config_name in [
        "configs/dailydialog_roberta.yaml",
        "configs/meld_roberta.yaml",
    ]:
        p = REPO_DIR / config_name
        cfg = yaml.safe_load(p.read_text(encoding="utf-8"))
        cfg["training"]["batch_size"] = int(BATCH_SIZE_OVERRIDE)
        p.write_text(
            yaml.safe_dump(
                cfg,
                sort_keys=False,
                allow_unicode=True,
            ),
            encoding="utf-8",
        )
        print(
            f"Set {config_name} "
            f"batch_size={BATCH_SIZE_OVERRIDE}"
        )

# Verify the packages actually required by this experiment.
print("\n=== REQUIRED PACKAGE CHECK ===")

import google.protobuf
import torch
import transformers
import pandas
import sklearn
import rdflib
import scipy
import accelerate
import yaml
import unidecode

print("protobuf    :", google.protobuf.__version__)
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("pandas      :", pandas.__version__)
print("sklearn     :", sklearn.__version__)
print("rdflib      :", rdflib.__version__)
print("scipy       :", scipy.__version__)
print("accelerate  :", accelerate.__version__)

from packaging.version import Version

pb_version = Version(google.protobuf.__version__)
assert (
    Version("5.29.5") <= pb_version < Version("6.0.0")
), (
    "Unexpected protobuf version: "
    f"{google.protobuf.__version__}"
)

print("\nDEPENDENCY CHECK PASSED.")
print(
    "Note: Kaggle may still report unrelated optional dependency warnings "
    "from preinstalled packages such as bigframes. They are not used by "
    "this experiment and do not affect this pipeline."
)

In [ ]:
# ============================================================
# 6. LOAD DAILYDIALOG FROM A VALIDATED PARQUET MIRROR
# ============================================================
#
# Why this change?
# The ACL attachment used in earlier notebook versions contains a malformed
# train record at dialogue 7851 in the extracted files available to this run:
# 12 utterances but only 11 emotion labels. We do NOT guess/truncate labels.
#
# Instead, we use the DeepPavlov/daily_dialog Hugging Face parquet mirror,
# which preserves the standard DailyDialog train/validation/test splits and
# exposes one utterance-level row with a scalar emotion label.
#
# Scientific dataset citation remains the original DailyDialog paper:
# Li et al. (2017), DailyDialog.
#
# We materialize clean Sentence/Emotion CSVs locally, and the repository's
# load_dailydialog() function will automatically use them.

import pandas as pd
import numpy as np

DD_DIR = DATA_ROOT / "dailydialog"
DD_DIR.mkdir(parents=True, exist_ok=True)

# Ensure Hugging Face datasets is available. Install only if missing.
try:
    from datasets import load_dataset
except ImportError:
    print("Installing Hugging Face datasets...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "datasets>=3.0,<5",
        ],
        check=True,
    )
    from datasets import load_dataset

HF_DD_REPO = "DeepPavlov/daily_dialog"

print("Loading DailyDialog parquet mirror:")
print(" ", HF_DD_REPO)

dd_hf = load_dataset(HF_DD_REPO)

print("\nAvailable splits:", list(dd_hf.keys()))
for split_name in dd_hf.keys():
    print(f"  {split_name:10s}: {len(dd_hf[split_name])}")

DD_EMOTION_MAP = {
    0: "Neutral",
    1: "Anger",
    2: "Disgust",
    3: "Fear",
    4: "Happiness",
    5: "Sadness",
    6: "Surprise",
}

DD_EMOTION_TEXT_MAP = {
    "no emotion": "Neutral",
    "neutral": "Neutral",
    "anger": "Anger",
    "disgust": "Disgust",
    "fear": "Fear",
    "happiness": "Happiness",
    "sadness": "Sadness",
    "surprise": "Surprise",
}

def _dialog_last_utterance(value):
    # DeepPavlov parquet stores cumulative dialogue context in `dialog`;
    # the final element is the current utterance represented by that row.
    if isinstance(value, np.ndarray):
        value = value.tolist()

    if isinstance(value, (list, tuple)):
        if len(value) == 0:
            return ""
        return str(value[-1]).strip()

    # Defensive fallback for unusual serialization.
    return str(value).strip()

def _convert_hf_split(ds_split, split_name):
    df = ds_split.to_pandas()

    required_base = {"dialog"}
    missing = required_base - set(df.columns)
    if missing:
        raise ValueError(
            f"DailyDialog mirror split {split_name} is missing "
            f"required columns: {sorted(missing)}"
        )

    # Prefer numeric canonical label if available.
    if "emotion_label" in df.columns:
        emotions = df["emotion_label"].map(DD_EMOTION_MAP)
    elif "emotion_label_text" in df.columns:
        emotions = (
            df["emotion_label_text"]
            .astype(str)
            .str.strip()
            .str.lower()
            .map(DD_EMOTION_TEXT_MAP)
        )
    else:
        raise ValueError(
            f"DailyDialog mirror split {split_name} has no "
            "`emotion_label` or `emotion_label_text` column."
        )

    out = pd.DataFrame({
        "Sentence": df["dialog"].map(_dialog_last_utterance),
        "Emotion": emotions,
    })

    if out["Emotion"].isna().any():
        bad_idx = out.index[out["Emotion"].isna()].tolist()[:10]
        raise ValueError(
            f"Unexpected DailyDialog emotion labels in split {split_name}; "
            f"example row indices: {bad_idx}"
        )

    # Do not silently delete records; fail if the mirror contains empty rows.
    empty_mask = out["Sentence"].astype(str).str.strip().eq("")
    if empty_mask.any():
        raise ValueError(
            f"DailyDialog mirror split {split_name} contains "
            f"{int(empty_mask.sum())} empty utterances."
        )

    return out.reset_index(drop=True)

dd_train_raw = _convert_hf_split(dd_hf["train"], "train")
dd_dev_raw = _convert_hf_split(dd_hf["validation"], "validation")
dd_test_raw = _convert_hf_split(dd_hf["test"], "test")

# Standard utterance-level counts for this DailyDialog representation.
EXPECTED_COUNTS = {
    "train": 87170,
    "validation": 8069,
    "test": 7740,
}

actual_counts = {
    "train": len(dd_train_raw),
    "validation": len(dd_dev_raw),
    "test": len(dd_test_raw),
}

print("\n=== DAILYDIALOG SPLIT CHECK ===")
for split_name, expected in EXPECTED_COUNTS.items():
    actual = actual_counts[split_name]
    print(
        f"{split_name:10s}: {actual:6d} utterances "
        f"(expected {expected:6d})"
    )
    if actual != expected:
        raise ValueError(
            f"DailyDialog {split_name} count mismatch: "
            f"{actual} != expected {expected}"
        )

# Save clean CSVs where load_dailydialog() can discover them recursively.
DD_CSV_DIR = DD_DIR / "normalized_csv"
DD_CSV_DIR.mkdir(parents=True, exist_ok=True)

dd_train_raw.to_csv(DD_CSV_DIR / "train.csv", index=False)
dd_dev_raw.to_csv(DD_CSV_DIR / "dev.csv", index=False)
dd_test_raw.to_csv(DD_CSV_DIR / "test.csv", index=False)

# Record provenance for reproducibility.
provenance = {
    "dataset": "DailyDialog",
    "original_citation": "Li et al. (2017), DailyDialog",
    "materialized_from": HF_DD_REPO,
    "representation": (
        "utterance-level rows; Sentence is the last item of the cumulative "
        "`dialog` field; original train/validation/test boundaries preserved"
    ),
    "label_mapping": DD_EMOTION_MAP,
    "counts": actual_counts,
}
(DD_DIR / "SOURCE_MANIFEST.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nDailyDialog normalized CSVs:")
for p in sorted(DD_CSV_DIR.glob("*.csv")):
    print(
        f"  {p.name:10s}",
        f"{p.stat().st_size / 1024**2:.2f} MB"
    )

print("\nClass distribution — train:")
print(dd_train_raw["Emotion"].value_counts().sort_index())

print("\nDailyDialog preparation PASSED.")

In [ ]:
# ============================================================
# 7. DOWNLOAD OFFICIAL MELD TEXT CSV FILES
# ============================================================

MELD_DIR = DATA_ROOT / "meld"
MELD_DIR.mkdir(parents=True, exist_ok=True)

MELD_BASE = (
    "https://raw.githubusercontent.com/"
    "declare-lab/MELD/master/data/MELD"
)

meld_names = [
    "train_sent_emo.csv",
    "dev_sent_emo.csv",
    "test_sent_emo.csv",
]

for name in meld_names:
    destination = MELD_DIR / name
    download_if_missing(f"{MELD_BASE}/{name}", destination)

print("\nMELD files:")
for p in sorted(MELD_DIR.glob("*.csv")):
    print(
        f"  {p.name:20s}",
        f"{p.stat().st_size / 1024:.1f} KB"
    )

for name in meld_names:
    if not (MELD_DIR / name).exists():
        raise RuntimeError(f"MELD file missing: {name}")

print("\nMELD download OK.")

In [ ]:
# ============================================================
# 8. BUILD NRC + ONTOLOGY-V2 ENGLISH LEXICAL LAYER
# ============================================================

os.chdir(REPO_DIR)

build_cmd = [
    sys.executable,
    "scripts/build_english_lexicon.py",
    "--nrc", str(NRC_PATH),
    "--ontology", "ontology/ekman_appraisal_ontology_v2.rdf",
    "--output", "ontology/english_nrc_ontology_v2_structured.json",
    "--flat-output", "ontology/english_nrc_flat7.json",
    "--summary-output",
    "ontology/english_nrc_ontology_v2_build_summary.json",
    "--confidence", "0.50",
]

print("Building NRC + ontology-v2 semantic resources...")
subprocess.run(build_cmd, check=True)

summary_path = (
    REPO_DIR
    / "ontology"
    / "english_nrc_ontology_v2_build_summary.json"
)
build_summary = json.loads(summary_path.read_text(encoding="utf-8"))

print("\n=== ONTOLOGY-V2 SEMANTIC MAPPING ===")
for emotion, info in build_summary["ontology_semantics"].items():
    print(
        f"{emotion:10s} "
        f"appraisals={info['appraisals']} "
        f"polarities={info['polarities']}"
    )

print("\nStructured keys:", build_summary["structured_keys"])
print(
    "Appraisal-enriched keys:",
    build_summary["appraisal_enriched_keys"],
)

assert (
    REPO_DIR
    / "ontology"
    / "english_nrc_ontology_v2_structured.json"
).exists()
assert (
    REPO_DIR / "ontology" / "english_nrc_flat7.json"
).exists()

# Core ontology-v2 appraisal mappings must exist.
for emotion in [
    "Anger",
    "Disgust",
    "Fear",
    "Happiness",
    "Sadness",
    "Surprise",
]:
    assert build_summary["ontology_semantics"][emotion]["appraisals"], (
        f"No appraisal semantics were derived for {emotion}. "
        "Check the ontology-v2 RDF."
    )

print("\nNRC + ontology-v2 lexical layer PASSED.")

In [ ]:
# ============================================================
# 9. DATASET + 24-D ONTOLOGY-V2 SANITY CHECKS
# ============================================================

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import pandas as pd
import numpy as np

from src.data_loading import load_dailydialog, load_meld
from src.language_adapters import EnglishAdapter
from src.ontology_engine import OntologyEngine

dd_train, dd_dev, dd_test = load_dailydialog(str(DD_DIR))
me_train, me_dev, me_test = load_meld(str(MELD_DIR))

print("=== DAILYDIALOG ===")
print("train/dev/test:", len(dd_train), len(dd_dev), len(dd_test))
print(dd_train["Emotion"].value_counts().sort_index())

print("\n=== MELD ===")
print("train/dev/test:", len(me_train), len(me_dev), len(me_test))
print(me_train["Emotion"].value_counts().sort_index())

expected_labels = {
    "Anger", "Disgust", "Fear", "Happiness",
    "Neutral", "Sadness", "Surprise"
}

for name, frame in [
    ("DailyDialog train", dd_train),
    ("DailyDialog dev", dd_dev),
    ("DailyDialog test", dd_test),
    ("MELD train", me_train),
    ("MELD dev", me_dev),
    ("MELD test", me_test),
]:
    bad = set(frame["Emotion"]) - expected_labels
    if bad:
        raise ValueError(f"{name} has unexpected labels: {sorted(bad)}")

engine = OntologyEngine(
    rdf_path=str(
        REPO_DIR / "ontology" / "ekman_appraisal_ontology_v2.rdf"
    ),
    default_conf=0.50,
    manual_boost=None,
    negation_attenuation=0.50,
    alpha_similarity=0.20,
    verbose=False,
    language_adapter=EnglishAdapter(),
    external_lexicon_path=str(
        REPO_DIR
        / "ontology"
        / "english_nrc_ontology_v2_structured.json"
    ),
)

test_sentences = [
    "I am angry and sad about this.",
    "I am not happy about this terrible result.",
]

for text in test_sentences:
    vec, tokens, trace = engine.getvector(text, debug=True)
    print("\nSentence:", text)
    print("Vector shape:", vec.shape)
    print("Tokens:", tokens)
    print("Emotion block   :", vec[0:7])
    print("Appraisal block :", vec[7:16])
    print("Intensity block :", vec[16:19])
    print("Polarity block  :", vec[19:22])
    print("Negation block  :", vec[22:24])
    print("Trace:", trace[:5])
    assert vec.shape == (24,)

# The appraisal branch must now be live for an emotion-bearing example.
probe = engine.getvector("I am angry and sad about this.")
assert np.any(np.abs(probe[7:16]) > 1e-8), (
    "Appraisal dimensions are still all zero. "
    "Do not run the pilot until this is fixed."
)

# Intensity remains deliberately uninferred without lexical evidence.
print("\nAppraisal dimensions are active: PASS")
print("Intensity is not inferred from NRC emotion alone: by design.")
print("\nSANITY CHECKS PASSED.")

In [ ]:
# ============================================================
# 10. SECOND PILOT — ONTOLOGY V2 + NEGATIVE CONTROLS
# ============================================================

def run_experiment(
    config,
    models,
    seeds,
    data_dir,
    output_root,
    rebuild_features=False,
):
    cmd = [
        sys.executable,
        "scripts/run_experiment.py",
        "--config", config,
        "--models", *models,
        "--seeds", *[str(x) for x in seeds],
        "--data-dir", str(data_dir),
        "--output-root", str(output_root),
    ]
    if rebuild_features:
        cmd.append("--rebuild-features")

    print("\nCOMMAND:")
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

def show_pilot_outputs(root, label):
    print("\n" + "=" * 80)
    print(label)
    print("=" * 80)

    summary = pd.read_csv(root / "summary.csv")
    display(summary[
        [
            "model",
            "n_seeds",
            "accuracy_mean",
            "macro_f1_mean",
            "ece_mean",
            "ontology_coverage_mean",
            "appraisal_coverage_mean",
            "gate_mean_mean_mean",
            "gated_ontology_norm_mean_mean",
        ]
    ])

    print("\nCoverage-stratified summary:")
    strata = pd.read_csv(root / "coverage_stratified_summary.csv")
    display(strata)

    print("\nFeature-group audit:")
    groups = pd.read_csv(root / "feature_group_audit.csv")
    display(groups)

if RUN_PILOT_V2:
    dd_pilot_v2 = RESULT_ROOT / "dailydialog_pilot_v2"
    me_pilot_v2 = RESULT_ROOT / "meld_pilot_v2"

    print("\n" + "=" * 80)
    print("DAILYDIALOG SECOND PILOT")
    print("=" * 80)
    run_experiment(
        "configs/dailydialog_roberta.yaml",
        PILOT_MODELS,
        PILOT_SEED,
        DD_DIR,
        dd_pilot_v2,
        rebuild_features=True,
    )
    show_pilot_outputs(dd_pilot_v2, "DailyDialog pilot-v2 summary")

    print("\n" + "=" * 80)
    print("MELD SECOND PILOT")
    print("=" * 80)
    run_experiment(
        "configs/meld_roberta.yaml",
        PILOT_MODELS,
        PILOT_SEED,
        MELD_DIR,
        me_pilot_v2,
        rebuild_features=True,
    )
    show_pilot_outputs(me_pilot_v2, "MELD pilot-v2 summary")

    print("\nSECOND PILOT COMPLETED.")
    print(
        "Do not enable five seeds yet. Download the pilot-v2 ZIP and "
        "review RawGate vs Zero/Shuffled first."
    )
else:
    print("Second pilot skipped because RUN_PILOT_V2=False.")

In [ ]:
# ============================================================
# 11. OPTIONAL CONFIRMATORY FIVE-SEED RUN
# ============================================================
#
# Keep disabled until pilot-v2 has been reviewed.

if RUN_CONFIRMATORY_5SEED:
    dd_full_v2 = RESULT_ROOT / "dailydialog_confirmatory_v2"
    me_full_v2 = RESULT_ROOT / "meld_confirmatory_v2"

    print("\nCONFIRMATORY DAILYDIALOG:")
    run_experiment(
        "configs/dailydialog_roberta.yaml",
        CONFIRMATORY_MODELS,
        FULL_SEEDS,
        DD_DIR,
        dd_full_v2,
        rebuild_features=True,
    )

    print("\nCONFIRMATORY MELD:")
    run_experiment(
        "configs/meld_roberta.yaml",
        CONFIRMATORY_MODELS,
        FULL_SEEDS,
        MELD_DIR,
        me_full_v2,
        rebuild_features=True,
    )

    print("\nConfirmatory five-seed runs completed.")
else:
    print(
        "Five-seed confirmatory experiments are OFF. "
        "This is intentional for the second pilot."
    )

In [ ]:
# ============================================================
# 12. PACKAGE SECOND-PILOT RESULTS
# ============================================================

archive_base = WORK_ROOT / "ijmlc_english_pilot_v2_results"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=RESULT_ROOT,
)

print("Results archive:", archive_path)
print(
    "Archive size (MB):",
    round(Path(archive_path).stat().st_size / 1024**2, 2)
)

print("\nPlease download this single file:")
print("/kaggle/working/ijmlc_english_pilot_v2_results.zip")

## What to do after pilot v2

Download:

`/kaggle/working/ijmlc_english_pilot_v2_results.zip`

The decision to proceed to five seeds should be based on four checks:

1. **Appraisal dimensions are active** in `feature_dimension_audit.csv`.
2. **RawGate(real) > RawGate-Zero** on the principal metrics.
3. **RawGate(real) > RawGate-Shuffled**, especially on ontology-active and
   appraisal-active samples.
4. DailyDialog no longer shows an unacceptable majority-class accuracy collapse.

Do not infer statistical significance from this one-seed pilot.  If these controls are
satisfactory, the next phase is a confirmatory multi-seed experiment followed by
paired resampling on saved predictions.